In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider

# Physical Parameters
M = 1.0     # Cart mass (kg)
m = 0.2     # Pendulum pole mass (kg)
L = 0.5     # Pole half-length (m)
g = 9.81    # Gravity (m/s^2)

def cartpole_dynamics(t, y, F):
    """Nonlinear cart-pole dynamics ODE system."""
    x, dx, theta, dtheta = y
    sin_th, cos_th = np.sin(theta), np.cos(theta)

    # Intermediate acceleration calculation
    temp = (F + m * L * dtheta**2 * sin_th) / (M + m)
    d2theta = (g * sin_th - cos_th * temp) / (L * (4.0/3.0 - (m * cos_th**2) / (M + m)))
    d2x = temp - (m * L * d2theta * cos_th) / (M + m)

    return [dx, d2x, dtheta, d2theta]

def simulate_cartpole(Kp, Ki, Kd, Kp_x, theta0_deg, t_final=5.0):
    dt = 0.01
    t_eval = np.arange(0, t_final, dt)

    # State initialization
    x, dx = 0.0, 0.0
    theta = np.radians(theta0_deg)
    dtheta = 0.0
    integral_e = 0.0
    Kd_x = 2.0 * np.sqrt(Kp_x) if Kp_x > 0 else 0.0  # Damping gain for cart position

    history_x, history_theta, history_F = [], [], []

    for t in t_eval:
        integral_e += theta * dt

        # Combined PID for angle + PD for cart position
        F = (Kp * theta + Kd * dtheta + Ki * integral_e) + (Kp_x * x + Kd_x * dx)
        F = np.clip(F, -150.0, 150.0) # Actuator saturation limit

        # Integrate forward one time step
        sol = solve_ivp(lambda t_curr, y_curr: cartpole_dynamics(t_curr, y_curr, F),
                        [t, t + dt], [x, dx, theta, dtheta], method='RK45')

        x, dx, theta, dtheta = sol.y[:, -1]

        history_x.append(x)
        history_theta.append(np.degrees(theta))
        history_F.append(F)

    return t_eval, np.array(history_x), np.array(history_theta), np.array(history_F)

def update_plot(Kp=120.0, Ki=5.0, Kd=25.0, Kp_x=8.0, Initial_Angle_deg=12.0):
    t_eval, x_hist, theta_hist, F_hist = simulate_cartpole(Kp, Ki, Kd, Kp_x, Initial_Angle_deg)

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8))

    # Plot 1: Visual Cart-Pole Frame (Final state)
    final_x = x_hist[-1]
    final_th = np.radians(theta_hist[-1])
    pole_x = final_x + (2 * L) * np.sin(final_th)
    pole_y = (2 * L) * np.cos(final_th)

    ax1.plot([-2, 2], [0, 0], 'k-', lw=2)  # Track
    ax1.plot([final_x - 0.2, final_x + 0.2], [0.05, 0.05], 's-b', lw=8, label='Cart')
    ax1.plot([final_x, pole_x], [0.05, pole_y], 'r-o', lw=4, label='Pendulum Pole')
    ax1.set_xlim([-2.5, 2.5])
    ax1.set_ylim([-0.5, 1.5])
    ax1.set_title(f"Cart-Pole Configuration (t={t_eval[-1]:.1f}s)")
    ax1.legend(loc='upper right')
    ax1.grid(True)

    # Plot 2: Pendulum Angle over Time
    ax2.plot(t_eval, theta_hist, 'r-', label=r'Angle $\theta$ (deg)')
    ax2.axhline(0, color='gray', linestyle='--')
    ax2.set_ylabel('Angle (deg)')
    ax2.set_title('Pendulum Angle Response')
    ax2.grid(True)
    ax2.legend()

    # Plot 3: Cart Position & Control Force over Time
    ax3.plot(t_eval, x_hist, 'b-', label='Position $x$ (m)')
    ax3.plot(t_eval, F_hist / 10.0, 'g--', label='Force $F / 10$ (N)')
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Amplitude')
    ax3.set_title('Cart Position and Control Input')
    ax3.grid(True)
    ax3.legend()

    plt.tight_layout()
    plt.show()

# Interactive Sliders for Colab
interact(update_plot,
         Kp=FloatSlider(min=0.0, max=300.0, step=5.0, value=120.0, description='Angle Kp'),
         Ki=FloatSlider(min=0.0, max=50.0, step=0.5, value=5.0, description='Angle Ki'),
         Kd=FloatSlider(min=0.0, max=100.0, step=1.0, value=25.0, description='Angle Kd'),
         Kp_x=FloatSlider(min=0.0, max=20.0, step=0.5, value=8.0, description='Cart Kp_x'),
         Initial_Angle_deg=FloatSlider(min=-30.0, max=30.0, step=1.0, value=12.0, description='Angle (deg)'));

interactive(children=(FloatSlider(value=120.0, description='Angle Kp', max=300.0, step=5.0), FloatSlider(value…

In [2]:
from IPython.display import HTML, display

html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Inverted Pendulum PID Control Simulator</title>
    <style>
        body { font-family: Arial, sans-serif; background-color: #1a1a2e; color: #e0e0e0; margin: 0; padding: 20px; }
        .container { max-width: 1050px; margin: 0 auto; background: #16213e; padding: 25px; border-radius: 12px; box-shadow: 0 8px 24px rgba(0,0,0,0.5); }
        h2 { text-align: center; color: #00fff5; margin-top: 0; }
        .flex-container { display: flex; flex-wrap: wrap; gap: 20px; justify-content: center; }
        .canvas-card { flex: 1; min-width: 480px; text-align: center; }
        canvas { background: #0f3460; border-radius: 8px; border: 2px solid #00fff5; }
        .controls-card { flex: 1; min-width: 320px; background: #0f3460; padding: 20px; border-radius: 8px; }
        .slider-group { margin-bottom: 14px; }
        .slider-group label { display: flex; justify-content: space-between; font-weight: bold; color: #e0e0e0; font-size: 14px; }
        input[type=range] { width: 100%; margin-top: 6px; accent-color: #00fff5; }
        .btn-group { display: flex; gap: 10px; margin-top: 20px; }
        button { flex: 1; padding: 10px; border: none; border-radius: 6px; font-weight: bold; cursor: pointer; color: white; transition: 0.2s; }
        .btn-reset { background: #e94560; } .btn-reset:hover { background: #ff2e63; }
        .btn-pause { background: #f39c12; } .btn-pause:hover { background: #d68910; }
        .telemetry { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; margin-top: 15px; background: #16213e; padding: 12px; border-radius: 6px; font-family: monospace; }
        .telemetry span { color: #00fff5; }
    </style>
</head>
<body>
<div class="container">
    <h2>Inverted Pendulum (Cart-Pole) PID Control</h2>
    <div class="flex-container">
        <div class="canvas-card">
            <canvas id="simCanvas" width="520" height="340"></canvas>
            <div class="telemetry">
                <div>Angle (&theta;): <span id="valTheta">0.0</span>&deg;</div>
                <div>Position (x): <span id="valX">0.00</span> m</div>
                <div>Force (F): <span id="valF">0.00</span> N</div>
                <div>State: <span id="valStatus" style="color:#00ff88;">Balancing</span></div>
            </div>
        </div>
        <div class="controls-card">
            <h3>Control Parameters</h3>
            <div class="slider-group">
                <label>Angle Kp: <span id="lblKp">120</span></label>
                <input type="range" id="kp" min="0" max="300" step="1" value="120">
            </div>
            <div class="slider-group">
                <label>Angle Ki: <span id="lblKi">5</span></label>
                <input type="range" id="ki" min="0" max="50" step="0.5" value="5">
            </div>
            <div class="slider-group">
                <label>Angle Kd: <span id="lblKd">25</span></label>
                <input type="range" id="kd" min="0" max="100" step="0.5" value="25">
            </div>
            <div class="slider-group">
                <label>Cart Position Gain (Kp_x): <span id="lblKpx">8</span></label>
                <input type="range" id="kpx" min="0" max="20" step="0.5" value="8">
            </div>
            <div class="slider-group">
                <label>Initial Angle (&deg;): <span id="lblAngle">12</span></label>
                <input type="range" id="initAngle" min="-30" max="30" step="1" value="12">
            </div>
            <div class="btn-group">
                <button class="btn-reset" onclick="resetSim()">Reset</button>
                <button class="btn-pause" id="btnPause" onclick="togglePause()">Pause</button>
            </div>
        </div>
    </div>
</div>

<script>
    const canvas = document.getElementById('simCanvas');
    const ctx = canvas.getContext('2d');

    const M = 1.0, m = 0.2, L = 0.5, g = 9.81, dt = 0.01;
    let x = 0, dx = 0, theta = 0.2, dtheta = 0, integral_e = 0;
    let isRunning = true;

    function resetSim() {
        x = 0; dx = 0;
        const angleDeg = parseFloat(document.getElementById('initAngle').value);
        theta = angleDeg * Math.PI / 180;
        dtheta = 0;
        integral_e = 0;
    }

    function togglePause() {
        isRunning = !isRunning;
        document.getElementById('btnPause').innerText = isRunning ? "Pause" : "Resume";
    }

    ['kp','ki','kd','kpx','initAngle'].forEach(id => {
        const el = document.getElementById(id);
        const lbl = document.getElementById('lbl' + id.charAt(0).toUpperCase() + id.slice(1));
        el.addEventListener('input', () => { if(lbl) lbl.innerText = el.value; });
    });

    function updatePhysics() {
        if(!isRunning) return;

        const Kp = parseFloat(document.getElementById('kp').value);
        const Ki = parseFloat(document.getElementById('ki').value);
        const Kd = parseFloat(document.getElementById('kd').value);
        const Kpx = parseFloat(document.getElementById('kpx').value);
        const Kdx = 2.0 * Math.sqrt(Kpx);

        integral_e += theta * dt;
        let F = (Kp * theta + Kd * dtheta + Ki * integral_e) + (Kpx * x + Kdx * dx);
        F = Math.max(-150, Math.min(150, F));

        const sinTh = Math.sin(theta), cosTh = Math.cos(theta);
        const temp = (F + m * L * dtheta * dtheta * sinTh) / (M + m);
        const d2theta = (g * sinTh - cosTh * temp) / (L * (4.0/3.0 - (m * cosTh * cosTh) / (M + m)));
        const d2x = temp - (m * L * d2theta * cosTh) / (M + m);

        dx += d2x * dt;
        x += dx * dt;
        dtheta += d2theta * dt;
        theta += dtheta * dt;

        document.getElementById('valTheta').innerText = (theta * 180 / Math.PI).toFixed(2);
        document.getElementById('valX').innerText = x.toFixed(2);
        document.getElementById('valF').innerText = F.toFixed(1);

        const isFallen = Math.abs(theta) > Math.PI / 3;
        document.getElementById('valStatus').innerText = isFallen ? "Unstable / Fallen" : "Balancing";
        document.getElementById('valStatus').style.color = isFallen ? "#e94560" : "#00ff88";
    }

    function draw() {
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        const cx = canvas.width / 2;
        const cy = canvas.height / 2 + 50;
        const scale = 90;

        // Draw Ground Track
        ctx.strokeStyle = '#4e54c8';
        ctx.lineWidth = 4;
        ctx.beginPath();
        ctx.moveTo(20, cy + 20);
        ctx.lineTo(canvas.width - 20, cy + 20);
        ctx.stroke();

        const cartX = cx + x * scale;
        const cartY = cy;
        const cartW = 75, cartH = 35;

        // Draw Cart
        ctx.fillStyle = '#00fff5';
        ctx.fillRect(cartX - cartW/2, cartY - cartH/2, cartW, cartH);

        // Draw Wheels
        ctx.fillStyle = '#111';
        ctx.beginPath();
        ctx.arc(cartX - 22, cartY + cartH/2, 8, 0, Math.PI*2);
        ctx.arc(cartX + 22, cartY + cartH/2, 8, 0, Math.PI*2);
        ctx.fill();

        // Draw Pendulum Pole
        const poleLen = (2 * L) * scale;
        const poleEndX = cartX + poleLen * Math.sin(theta);
        const poleEndY = cartY - poleLen * Math.cos(theta);

        ctx.strokeStyle = '#e94560';
        ctx.lineWidth = 6;
        ctx.lineCap = 'round';
        ctx.beginPath();
        ctx.moveTo(cartX, cartY);
        ctx.lineTo(poleEndX, poleEndY);
        ctx.stroke();

        // Pivot Point
        ctx.fillStyle = '#fff';
        ctx.beginPath();
        ctx.arc(cartX, cartY, 5, 0, Math.PI*2);
        ctx.fill();

        // Bob Mass
        ctx.fillStyle = '#ff2e63';
        ctx.beginPath();
        ctx.arc(poleEndX, poleEndY, 10, 0, Math.PI*2);
        ctx.fill();
    }

    function loop() {
        updatePhysics();
        draw();
        requestAnimationFrame(loop);
    }
    loop();
</script>
</body>
</html>
"""

with open("cart_pole_pid.html", "w") as f:
    f.write(html_content)

print("File 'cart_pole_pid.html' created successfully!")
display(HTML(html_content))

File 'cart_pole_pid.html' created successfully!


In [3]:
from google.colab import files
files.download('cart_pole_pid.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>